# Vision Inspection Portfolio - Training Notebook v2

## Purpose
Complete YOLOv8 training pipeline for surface defect detection using MVTec AD dataset.
Supports bottle and tile categories with automated data preparation and model training.

## Usage Instructions
1. Upload this notebook to Google Colab
2. Enable GPU runtime (Runtime > Change runtime type > T4 GPU)
3. Run all cells sequentially
4. Monitor training progress and final mAP50 scores

## Google Drive Structure
```
vision_portfolio/
├── mvtec/
│   ├── tile/tile/         <- tile MVTec raw data (extracted)
│   └── bottle/            <- bottle MVTec raw data (to be extracted)
├── bottle.zip             <- bottle MVTec raw data
├── yolo/                  <- YOLO format datasets (generated)
│   ├── bottle/
│   └── tile/
└── runs/                  <- training outputs
    ├── bottle_n/
    ├── bottle_s/
    └── tile_n/
```

## Model Categories
- **Bottle**: broken_large, broken_small, contamination
- **Tile**: crack, glue_strip, gray_stroke, oil, rough

In [ ]:
# Mount Google Drive and verify vision_portfolio folder
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Set base path
base_path = '/content/drive/MyDrive/vision_portfolio'

# Verify vision_portfolio folder exists
if os.path.exists(base_path):
    print(f"✓ Found vision_portfolio at: {base_path}")
    print("\nDirectory contents:")
    for item in os.listdir(base_path):
        item_path = os.path.join(base_path, item)
        if os.path.isdir(item_path):
            print(f"  📁 {item}/")
        else:
            print(f"  📄 {item}")
else:
    print(f"❌ vision_portfolio folder not found at: {base_path}")
    print("Please create the folder structure in Google Drive first.")

In [ ]:
# Extract bottle.zip to mvtec folder
import zipfile
import os

# Define paths
bottle_zip_path = '/content/drive/MyDrive/vision_portfolio/bottle.zip'
extract_to_path = '/content/drive/MyDrive/vision_portfolio/mvtec/'
bottle_extracted_path = '/content/drive/MyDrive/vision_portfolio/mvtec/bottle'

# Check if already extracted
if os.path.exists(bottle_extracted_path):
    print(f"✓ Bottle dataset already extracted at: {bottle_extracted_path}")
else:
    # Create mvtec directory if it doesn't exist
    os.makedirs(extract_to_path, exist_ok=True)
    
    # Extract bottle.zip
    if os.path.exists(bottle_zip_path):
        print(f"Extracting {bottle_zip_path} to {extract_to_path}...")
        with zipfile.ZipFile(bottle_zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_to_path)
        print("✓ Extraction completed")
    else:
        print(f"❌ bottle.zip not found at: {bottle_zip_path}")

# Verify extraction and print folder structure
if os.path.exists(bottle_extracted_path):
    print(f"\nBottle dataset structure:")
    for root, dirs, files in os.walk(bottle_extracted_path):
        level = root.replace(bottle_extracted_path, '').count(os.sep)
        indent = ' ' * 2 * level
        folder_name = os.path.basename(root)
        print(f"{indent}{folder_name}/ ({len(files)} files)")

In [ ]:
# Convert bottle dataset to YOLO format
import os
import cv2
import numpy as np
import shutil
from sklearn.model_selection import train_test_split
import yaml
import random

# Set random seed for reproducibility
random.seed(42)
np.random.seed(42)

def mask_to_bbox(mask_path):
    """Convert segmentation mask to YOLO bounding box format"""
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return None
    
    # Threshold mask to binary
    _, binary_mask = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
    
    # Find contours
    contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    if not contours:
        return None
    
    # Get bounding box of largest contour
    largest_contour = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(largest_contour)
    
    # Convert to YOLO format (normalized)
    img_h, img_w = mask.shape
    x_center = (x + w/2) / img_w
    y_center = (y + h/2) / img_h
    width = w / img_w
    height = h / img_h
    
    return x_center, y_center, width, height

# Define paths
bottle_raw_path = '/content/drive/MyDrive/vision_portfolio/mvtec/bottle'
bottle_yolo_path = '/content/drive/MyDrive/vision_portfolio/yolo/bottle'

# Define classes
bottle_classes = ['broken_large', 'broken_small', 'contamination']
class_to_id = {cls: idx for idx, cls in enumerate(bottle_classes)}

# Create YOLO directory structure
for split in ['train', 'val']:
    for folder in ['images', 'labels']:
        os.makedirs(os.path.join(bottle_yolo_path, split, folder), exist_ok=True)

# Process good images (all go to train)
good_train_path = os.path.join(bottle_raw_path, 'train', 'good')
train_count = 0
val_count = 0

if os.path.exists(good_train_path):
    for img_file in os.listdir(good_train_path):
        if img_file.endswith('.png'):
            # Copy image to train
            src_img = os.path.join(good_train_path, img_file)
            dst_img = os.path.join(bottle_yolo_path, 'train', 'images', img_file)
            shutil.copy2(src_img, dst_img)
            
            # Create empty label file (no defects)
            label_file = img_file.replace('.png', '.txt')
            dst_label = os.path.join(bottle_yolo_path, 'train', 'labels', label_file)
            with open(dst_label, 'w') as f:
                pass  # Empty file for good images
            
            train_count += 1

# Process defect images (80% train / 20% val)
for defect_class in bottle_classes:
    test_defect_path = os.path.join(bottle_raw_path, 'test', defect_class)
    gt_defect_path = os.path.join(bottle_raw_path, 'ground_truth', defect_class)
    
    if os.path.exists(test_defect_path) and os.path.exists(gt_defect_path):
        defect_images = [f for f in os.listdir(test_defect_path) if f.endswith('.png')]
        
        # Split 80/20
        train_defects, val_defects = train_test_split(
            defect_images, test_size=0.2, random_state=42, shuffle=True
        )
        
        # Process train defects
        for img_file in train_defects:
            # Copy image
            src_img = os.path.join(test_defect_path, img_file)
            dst_img = os.path.join(bottle_yolo_path, 'train', 'images', img_file)
            shutil.copy2(src_img, dst_img)
            
            # Convert mask to YOLO label
            mask_file = img_file  # Same filename as image
            mask_path = os.path.join(gt_defect_path, mask_file)
            
            if os.path.exists(mask_path):
                bbox = mask_to_bbox(mask_path)
                if bbox:
                    label_file = img_file.replace('.png', '.txt')
                    dst_label = os.path.join(bottle_yolo_path, 'train', 'labels', label_file)
                    
                    with open(dst_label, 'w') as f:
                        class_id = class_to_id[defect_class]
                        f.write(f"{class_id} {bbox[0]:.6f} {bbox[1]:.6f} {bbox[2]:.6f} {bbox[3]:.6f}\n")
            
            train_count += 1
        
        # Process val defects
        for img_file in val_defects:
            # Copy image
            src_img = os.path.join(test_defect_path, img_file)
            dst_img = os.path.join(bottle_yolo_path, 'val', 'images', img_file)
            shutil.copy2(src_img, dst_img)
            
            # Convert mask to YOLO label
            mask_file = img_file
            mask_path = os.path.join(gt_defect_path, mask_file)
            
            if os.path.exists(mask_path):
                bbox = mask_to_bbox(mask_path)
                if bbox:
                    label_file = img_file.replace('.png', '.txt')
                    dst_label = os.path.join(bottle_yolo_path, 'val', 'labels', label_file)
                    
                    with open(dst_label, 'w') as f:
                        class_id = class_to_id[defect_class]
                        f.write(f"{class_id} {bbox[0]:.6f} {bbox[1]:.6f} {bbox[2]:.6f} {bbox[3]:.6f}\n")
            
            val_count += 1

# Create dataset.yaml with Colab absolute paths
dataset_config = {
    'path': '/content/drive/MyDrive/vision_portfolio/yolo/bottle',
    'train': '/content/drive/MyDrive/vision_portfolio/yolo/bottle/train/images',
    'val': '/content/drive/MyDrive/vision_portfolio/yolo/bottle/val/images',
    'nc': len(bottle_classes),
    'names': bottle_classes
}

with open(os.path.join(bottle_yolo_path, 'dataset.yaml'), 'w') as f:
    yaml.dump(dataset_config, f, default_flow_style=False)

print(f"✓ Bottle dataset converted to YOLO format")
print(f"Train images: {train_count}")
print(f"Validation images: {val_count}")
print(f"Classes: {bottle_classes}")
print(f"Dataset saved to: {bottle_yolo_path}")

In [ ]:
# Convert tile dataset to YOLO format
import os
import cv2
import numpy as np
import shutil
from sklearn.model_selection import train_test_split
import yaml
import random

# Set random seed for reproducibility
random.seed(42)
np.random.seed(42)

# Define paths
tile_raw_path = '/content/drive/MyDrive/vision_portfolio/mvtec/tile/tile'
tile_yolo_path = '/content/drive/MyDrive/vision_portfolio/yolo/tile'

# Define classes
tile_classes = ['crack', 'glue_strip', 'gray_stroke', 'oil', 'rough']
class_to_id = {cls: idx for idx, cls in enumerate(tile_classes)}

# Create YOLO directory structure
for split in ['train', 'val']:
    for folder in ['images', 'labels']:
        os.makedirs(os.path.join(tile_yolo_path, split, folder), exist_ok=True)

# Process good images (all go to train)
good_train_path = os.path.join(tile_raw_path, 'train', 'good')
train_count = 0
val_count = 0

if os.path.exists(good_train_path):
    for img_file in os.listdir(good_train_path):
        if img_file.endswith('.png'):
            # Copy image to train
            src_img = os.path.join(good_train_path, img_file)
            dst_img = os.path.join(tile_yolo_path, 'train', 'images', img_file)
            shutil.copy2(src_img, dst_img)
            
            # Create empty label file (no defects)
            label_file = img_file.replace('.png', '.txt')
            dst_label = os.path.join(tile_yolo_path, 'train', 'labels', label_file)
            with open(dst_label, 'w') as f:
                pass  # Empty file for good images
            
            train_count += 1

# Process defect images (80% train / 20% val)
for defect_class in tile_classes:
    test_defect_path = os.path.join(tile_raw_path, 'test', defect_class)
    gt_defect_path = os.path.join(tile_raw_path, 'ground_truth', defect_class)
    
    if os.path.exists(test_defect_path) and os.path.exists(gt_defect_path):
        defect_images = [f for f in os.listdir(test_defect_path) if f.endswith('.png')]
        
        # Split 80/20
        train_defects, val_defects = train_test_split(
            defect_images, test_size=0.2, random_state=42, shuffle=True
        )
        
        # Process train defects
        for img_file in train_defects:
            # Copy image
            src_img = os.path.join(test_defect_path, img_file)
            dst_img = os.path.join(tile_yolo_path, 'train', 'images', img_file)
            shutil.copy2(src_img, dst_img)
            
            # Convert mask to YOLO label
            mask_file = img_file  # Same filename as image
            mask_path = os.path.join(gt_defect_path, mask_file)
            
            if os.path.exists(mask_path):
                bbox = mask_to_bbox(mask_path)
                if bbox:
                    label_file = img_file.replace('.png', '.txt')
                    dst_label = os.path.join(tile_yolo_path, 'train', 'labels', label_file)
                    
                    with open(dst_label, 'w') as f:
                        class_id = class_to_id[defect_class]
                        f.write(f"{class_id} {bbox[0]:.6f} {bbox[1]:.6f} {bbox[2]:.6f} {bbox[3]:.6f}\n")
            
            train_count += 1
        
        # Process val defects
        for img_file in val_defects:
            # Copy image
            src_img = os.path.join(test_defect_path, img_file)
            dst_img = os.path.join(tile_yolo_path, 'val', 'images', img_file)
            shutil.copy2(src_img, dst_img)
            
            # Convert mask to YOLO label
            mask_file = img_file
            mask_path = os.path.join(gt_defect_path, mask_file)
            
            if os.path.exists(mask_path):
                bbox = mask_to_bbox(mask_path)
                if bbox:
                    label_file = img_file.replace('.png', '.txt')
                    dst_label = os.path.join(tile_yolo_path, 'val', 'labels', label_file)
                    
                    with open(dst_label, 'w') as f:
                        class_id = class_to_id[defect_class]
                        f.write(f"{class_id} {bbox[0]:.6f} {bbox[1]:.6f} {bbox[2]:.6f} {bbox[3]:.6f}\n")
            
            val_count += 1

# Create dataset.yaml with Colab absolute paths
dataset_config = {
    'path': '/content/drive/MyDrive/vision_portfolio/yolo/tile',
    'train': '/content/drive/MyDrive/vision_portfolio/yolo/tile/train/images',
    'val': '/content/drive/MyDrive/vision_portfolio/yolo/tile/val/images',
    'nc': len(tile_classes),
    'names': tile_classes
}

with open(os.path.join(tile_yolo_path, 'dataset.yaml'), 'w') as f:
    yaml.dump(dataset_config, f, default_flow_style=False)

print(f"✓ Tile dataset converted to YOLO format")
print(f"Train images: {train_count}")
print(f"Validation images: {val_count}")
print(f"Classes: {tile_classes}")
print(f"Dataset saved to: {tile_yolo_path}")

In [ ]:
# Install required dependencies
import subprocess
import sys

# Install ultralytics and onnx
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'ultralytics', 'onnx'])

# Import and check versions
import ultralytics
import onnx
import torch

print(f"✓ Ultralytics version: {ultralytics.__version__}")
print(f"✓ ONNX version: {onnx.__version__}")
print(f"✓ PyTorch version: {torch.__version__}")
print(f"✓ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name()}")

In [ ]:
# Train bottle YOLOv8n model
from ultralytics import YOLO
import os
import shutil

print("Starting bottle YOLOv8n training...")

# Initialize model
model = YOLO('yolov8n.pt')

# Train to local storage first for speed
results = model.train(
    data='/content/drive/MyDrive/vision_portfolio/yolo/bottle/dataset.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    project='/content/runs',
    name='bottle_n',
    exist_ok=True,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    flipud=0.3,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    save=True,
    plots=True
)

# Get final mAP50 score
final_map50 = results.results_dict.get('metrics/mAP50(B)', 'N/A')
print(f"\nBottle YOLOv8n training completed")
print(f"Final mAP50: {final_map50}")

# Copy results to Google Drive
local_path = '/content/runs/bottle_n'
drive_path = '/content/drive/MyDrive/vision_portfolio/runs/bottle_n'

if os.path.exists(local_path):
    # Create drive directory if needed
    os.makedirs('/content/drive/MyDrive/vision_portfolio/runs', exist_ok=True)
    
    # Remove existing drive folder if it exists
    if os.path.exists(drive_path):
        shutil.rmtree(drive_path)
    
    # Copy to drive
    shutil.copytree(local_path, drive_path)
    print(f"Results copied to: {drive_path}")
else:
    print("Warning: Local training results not found")

In [ ]:
# Train bottle YOLOv8s model
from ultralytics import YOLO
import os
import shutil

print("Starting bottle YOLOv8s training...")

# Initialize model
model = YOLO('yolov8s.pt')

# Train to local storage first for speed
results = model.train(
    data='/content/drive/MyDrive/vision_portfolio/yolo/bottle/dataset.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    project='/content/runs',
    name='bottle_s',
    exist_ok=True,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    flipud=0.3,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    save=True,
    plots=True
)

# Get final mAP50 score
final_map50 = results.results_dict.get('metrics/mAP50(B)', 'N/A')
print(f"\nBottle YOLOv8s training completed")
print(f"Final mAP50: {final_map50}")

# Copy results to Google Drive
local_path = '/content/runs/bottle_s'
drive_path = '/content/drive/MyDrive/vision_portfolio/runs/bottle_s'

if os.path.exists(local_path):
    # Create drive directory if needed
    os.makedirs('/content/drive/MyDrive/vision_portfolio/runs', exist_ok=True)
    
    # Remove existing drive folder if it exists
    if os.path.exists(drive_path):
        shutil.rmtree(drive_path)
    
    # Copy to drive
    shutil.copytree(local_path, drive_path)
    print(f"Results copied to: {drive_path}")
else:
    print("Warning: Local training results not found")

In [ ]:
# Train tile YOLOv8n model
from ultralytics import YOLO
import os
import shutil

print("Starting tile YOLOv8n training...")

# Initialize model
model = YOLO('yolov8n.pt')

# Train to local storage first for speed
results = model.train(
    data='/content/drive/MyDrive/vision_portfolio/yolo/tile/dataset.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    project='/content/runs',
    name='tile_n',
    exist_ok=True,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    flipud=0.3,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    save=True,
    plots=True
)

# Get final mAP50 score
final_map50 = results.results_dict.get('metrics/mAP50(B)', 'N/A')
print(f"\nTile YOLOv8n training completed")
print(f"Final mAP50: {final_map50}")

# Copy results to Google Drive
local_path = '/content/runs/tile_n'
drive_path = '/content/drive/MyDrive/vision_portfolio/runs/tile_n'

if os.path.exists(local_path):
    # Create drive directory if needed
    os.makedirs('/content/drive/MyDrive/vision_portfolio/runs', exist_ok=True)
    
    # Remove existing drive folder if it exists
    if os.path.exists(drive_path):
        shutil.rmtree(drive_path)
    
    # Copy to drive
    shutil.copytree(local_path, drive_path)
    print(f"Results copied to: {drive_path}")
else:
    print("Warning: Local training results not found")

In [ ]:
# Export all trained models to ONNX format with opset=21
from ultralytics import YOLO
import os
import onnx
import shutil

# Define model paths (load from local storage)
models_to_export = [
    {
        'name': 'bottle_n',
        'local_path': '/content/runs/bottle_n/weights/best.pt',
        'drive_path': '/content/drive/MyDrive/vision_portfolio/runs/bottle_n/weights/best.onnx'
    },
    {
        'name': 'bottle_s',
        'local_path': '/content/runs/bottle_s/weights/best.pt',
        'drive_path': '/content/drive/MyDrive/vision_portfolio/runs/bottle_s/weights/best.onnx'
    },
    {
        'name': 'tile_n',
        'local_path': '/content/runs/tile_n/weights/best.pt',
        'drive_path': '/content/drive/MyDrive/vision_portfolio/runs/tile_n/weights/best.onnx'
    }
]

print("Exporting models to ONNX format with opset=21...\n")

for model_info in models_to_export:
    model_name = model_info['name']
    local_pt_path = model_info['local_path']
    drive_onnx_path = model_info['drive_path']
    
    if os.path.exists(local_pt_path):
        print(f"Exporting {model_name}...")
        
        # Load model from local storage
        model = YOLO(local_pt_path)
        
        # Export to ONNX with opset=21, simplify=True, imgsz=640
        local_onnx_path = model.export(format='onnx', opset=21, simplify=True, imgsz=640)
        
        # Verify opset version
        onnx_model = onnx.load(local_onnx_path)
        opset_version = onnx_model.opset_import[0].version
        
        # Get file size
        file_size = os.path.getsize(local_onnx_path) / (1024 * 1024)  # MB
        
        # Copy ONNX file to Google Drive
        os.makedirs(os.path.dirname(drive_onnx_path), exist_ok=True)
        shutil.copy2(local_onnx_path, drive_onnx_path)
        
        print(f"  Exported successfully")
        print(f"  Local path: {local_onnx_path}")
        print(f"  Drive path: {drive_onnx_path}")
        print(f"  Opset version: {opset_version}")
        print(f"  File size: {file_size:.2f} MB\n")
    else:
        print(f"  Model not found: {local_pt_path}\n")

print("All model exports completed")

In [ ]:
# Validate all exported ONNX models with defect detection
from ultralytics import YOLO
import os
import glob
import random

# Set random seed
random.seed(42)

print("Validating exported ONNX models...\n")

# Define model information with test images
models_to_validate = [
    {
        'name': 'bottle_n',
        'onnx_path': '/content/drive/MyDrive/vision_portfolio/runs/bottle_n/weights/best.onnx',
        'test_images': {
            'broken_large': '/content/drive/MyDrive/vision_portfolio/mvtec/bottle/test/broken_large',
            'broken_small': '/content/drive/MyDrive/vision_portfolio/mvtec/bottle/test/broken_small',
            'contamination': '/content/drive/MyDrive/vision_portfolio/mvtec/bottle/test/contamination'
        }
    },
    {
        'name': 'bottle_s',
        'onnx_path': '/content/drive/MyDrive/vision_portfolio/runs/bottle_s/weights/best.onnx',
        'test_images': {
            'broken_large': '/content/drive/MyDrive/vision_portfolio/mvtec/bottle/test/broken_large',
            'broken_small': '/content/drive/MyDrive/vision_portfolio/mvtec/bottle/test/broken_small',
            'contamination': '/content/drive/MyDrive/vision_portfolio/mvtec/bottle/test/contamination'
        }
    },
    {
        'name': 'tile_n',
        'onnx_path': '/content/drive/MyDrive/vision_portfolio/runs/tile_n/weights/best.onnx',
        'test_images': {
            'crack': '/content/drive/MyDrive/vision_portfolio/mvtec/tile/tile/test/crack',
            'glue_strip': '/content/drive/MyDrive/vision_portfolio/mvtec/tile/tile/test/glue_strip',
            'gray_stroke': '/content/drive/MyDrive/vision_portfolio/mvtec/tile/tile/test/gray_stroke',
            'oil': '/content/drive/MyDrive/vision_portfolio/mvtec/tile/tile/test/oil',
            'rough': '/content/drive/MyDrive/vision_portfolio/mvtec/tile/tile/test/rough'
        }
    }
]

confidence_threshold = 0.25

for model_info in models_to_validate:
    model_name = model_info['name']
    onnx_path = model_info['onnx_path']
    test_categories = model_info['test_images']
    
    if os.path.exists(onnx_path):
        print(f"Validating {model_name}...")
        
        # Load ONNX model
        model = YOLO(onnx_path)
        
        # Test one image per category
        for category, test_dir in test_categories.items():
            if os.path.exists(test_dir):
                # Get random test image
                test_images = [f for f in os.listdir(test_dir) if f.endswith('.png')]
                if test_images:
                    test_image = random.choice(test_images)
                    test_image_path = os.path.join(test_dir, test_image)
                    
                    # Run inference
                    results = model(test_image_path, conf=confidence_threshold, verbose=False)
                    
                    # Analyze results
                    if results and len(results) > 0:
                        detections = results[0].boxes
                        if detections is not None and len(detections) > 0:
                            max_conf = float(detections.conf.max())
                            detection_count = len(detections)
                            
                            status = "✓ PASS" if max_conf >= confidence_threshold else "❌ FAIL"
                            print(f"  {category}: {status} - {detection_count} detections, max conf: {max_conf:.3f}")
                        else:
                            print(f"  {category}: ❌ FAIL - No detections above threshold {confidence_threshold}")
                    else:
                        print(f"  {category}: ❌ FAIL - No inference results")
                else:
                    print(f"  {category}: ⚠️ SKIP - No test images found")
            else:
                print(f"  {category}: ⚠️ SKIP - Test directory not found")
        
        print()  # Empty line for readability
    else:
        print(f"❌ ONNX model not found: {onnx_path}\n")

print("✓ Model validation completed")
print(f"All models should detect defects with confidence >= {confidence_threshold}")